# AI-Assisted Interview Feedback & Communication Analysis
## Exploratory Data Analysis & NLP Model Benchmarking
**TY B.Sc. Data Science Project**
**Student:** Srushti Dhide (Roll No. 54)

This notebook provides rigorous data science exploratory analysis, linguistic metrics benchmarking, and statistical distribution modeling for the automated mock interview platform.

### 1. Environment Setup & Data Ingestion

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
sys.path.append(os.path.abspath('..'))
from modules import nlp_analysis, relevance, communication, scoring

# Set visual theme
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('Libraries and custom modules successfully imported!')

### 2. Question Bank Dataset Exploration

In [ ]:
df_questions = pd.read_csv('../data/questions.csv')
print(f'Total Questions: {len(df_questions)}')
df_questions.head()

In [ ]:
# Category and Difficulty Breakdown
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

df_questions['category'].value_counts().plot(kind='bar', ax=ax1, color=['#3B82F6', '#10B981'])
ax1.set_title('Questions by Category', fontsize=12, fontweight='bold')
ax1.set_ylabel('Count')

df_questions['difficulty'].value_counts().plot(kind='pie', ax=ax2, autopct='%1.1f%%', colors=['#60A5FA', '#FBBF24', '#F87171'])
ax2.set_title('Difficulty Distribution', fontsize=12, fontweight='bold')
ax2.set_ylabel('')

plt.tight_layout()
plt.show()

### 3. Filler Words Distribution Analysis

In [ ]:
fillers = nlp_analysis.load_filler_words('../data/filler_words.txt')
print(f'Total configured filler terms/phrases: {len(fillers)}')
print('Sample fillers:', fillers[:10])

# Simulate a sample candidate answer with various fillers
sample_text = (
    "Um, basically, machine learning is, like, a branch of artificial intelligence where, "
    "you know, models learn patterns from data, sort of without explicit programming. "
    "Actually, to be honest, supervised algorithms need labels."
)

detected = nlp_analysis.detect_filler_words(sample_text, fillers)
print(f"Total Fillers: {detected['filler_count']}")
print(f"Filler Frequency: {detected['filler_frequency']}% of total words")
print("Breakdown:", detected['fillers_found'])

### 4. TF-IDF & Cosine Similarity Relevance Modeling
Evaluating semantic similarity behavior between candidate answers and reference answers.

In [ ]:
qid = 21
q_row = df_questions[df_questions['question_id'] == qid].iloc[0].to_dict()
print('Question:', q_row['question'])

candidates = [
    ('Strong Answer', 'Supervised learning algorithms train on labeled data for classification and regression tasks, while unsupervised learning algorithms discover intrinsic clusters in unlabeled data.'),
    ('Partial Answer', 'Supervised has labels and unsupervised does not have labels.'),
    ('Off-Topic Answer', 'I really enjoy drinking black coffee while hiking in the Western Ghats during the monsoon season.')
]

results = []
for label, ans in candidates:
    eval_res = relevance.evaluate_relevance(ans, q_row)
    results.append({
        'Answer Type': label,
        'Relevance Score': eval_res['relevance_score'],
        'TF-IDF Cosine (Ref)': eval_res['cosine_similarity_ref'],
        'Keyword Coverage': eval_res['keyword_coverage_ratio'],
        'Rating': eval_res['relevance_rating']
    })

df_results = pd.DataFrame(results)
df_results

### 5. Deterministic Scoring Simulation & Sensitivity Analysis
Verifying the composite weights configuration across diverse student responses.

In [ ]:
print('Configured Scoring Weights:')
for k, v in scoring.SCORE_WEIGHTS.items():
    print(f'  - {k.capitalize()}: {v * 100}%')

# Simulate 100 student responses with random component variations
np.random.seed(42)
sim_relevance = np.random.normal(70, 15, 100).clip(20, 100)
sim_fluency = np.random.normal(75, 12, 100).clip(30, 100)
sim_vocab = np.random.normal(68, 14, 100).clip(25, 100)
sim_grammar = np.random.normal(80, 10, 100).clip(40, 100)
sim_filler_freq = np.random.exponential(2.0, 100).clip(0, 10)

total_scores = []
for r, f, v, g, ff in zip(sim_relevance, sim_fluency, sim_vocab, sim_grammar, sim_filler_freq):
    score = scoring.calculate_response_score(r, f, v, g, ff)
    total_scores.append(score['total_score'])

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(total_scores, kde=True, bins=15, color='#2563EB', ax=ax)
ax.set_title('Simulated Interview Score Distribution (n=100)', fontsize=12, fontweight='bold')
ax.set_xlabel('Overall Score (0 - 100)')
ax.set_ylabel('Candidate Frequency')
plt.tight_layout()
plt.show()

print(f'Mean Score: {np.mean(total_scores):.1f}')
print(f'Standard Deviation: {np.std(total_scores):.1f}')
print(f'Median Score: {np.median(total_scores):.1f}')